# NeuroScope — training & evaluation on BraTS 2020 (Kaggle)

**Before running:** right sidebar → *Accelerator: GPU P100* (or T4 x2), *Internet: On*, and the dataset
`awsaf49/brats20-dataset-training-validation` attached under *Input*.

Run cells top to bottom. Total ≈ 60–80 min on a P100: preprocess ~10 min · train ~25 min · evaluate ~30 min.
If the session dies, every step is resumable (cache and checkpoints live in `/kaggle/working`).

In [ ]:
# 1) Get the code. EITHER clone your GitHub repo ...
REPO_URL = "https://github.com/neetii-git/neuroscope.git"   
import os
if not os.path.isdir("/kaggle/working/neuroscope"):
    !git clone -q {REPO_URL} /kaggle/working/neuroscope
# ... OR, if you uploaded neuroscope.zip as a Kaggle dataset instead, comment the clone above and use:
# !unzip -q -o /kaggle/input/neuroscope-code/neuroscope.zip -d /kaggle/working/
%cd /kaggle/working/neuroscope
!pip install -q nibabel pyyaml 2>/dev/null | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Unit tests (synthetic data, ~5 s) — proves the pipeline before touching real MRI
!python -m pytest -q tests

In [ ]:
# 3) Preprocess all 369 training cases -> uint8 cache + patient-level split (train 258 / val 37 / test 74)
RAW   = "/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
CACHE = "/kaggle/working/cache"
!ls {RAW} | head -3
!python scripts/preprocess.py --raw {RAW} --cache {CACHE}
!du -sh {CACHE}; python -c "import json;s=json.load(open('/kaggle/working/cache/split.json'));print({k:len(v) for k,v in s.items() if isinstance(v,list)})" 

In [ ]:
# 4) Train the modality-robust U-Net (modality dropout p=0.5, MC-dropout layers, Dice+BCE, AMP)
#    ~1 min/epoch on P100. Raise --epochs to 30 if you have time; lower to 12 if you don't.
!python scripts/train.py --cache /kaggle/working/cache --out /kaggle/working/runs/base --epochs 20
import pandas as pd; pd.read_csv("/kaggle/working/runs/base/history.csv").tail()

In [ ]:
# 5) Evaluate: test metrics (+95% CI), 15 modality subsets, uncertainty reliability, review threshold (chosen on val), corruptions
!python scripts/evaluate.py --cache /kaggle/working/cache --ckpt /kaggle/working/runs/base/best.pt --out /kaggle/working/runs/base/results
import json; print(json.dumps(json.load(open("/kaggle/working/runs/base/results/summary.json")), indent=1)[:3000])

In [ ]:
# 6) Look at the results
import pandas as pd
from IPython.display import Image, display
R = "/kaggle/working/runs/base/results"
display(pd.read_csv(f"{R}/subsets.csv").round(3))
display(pd.read_csv(f"{R}/corruptions.csv").round(3))
for f in ["fig_subsets.png", "fig_uncertainty_vs_dice.png", "fig_risk_coverage.png"]:
    display(Image(f"{R}/{f}"))

In [ ]:
# 7) OPTIONAL ABLATION (+~35 min): same model trained WITHOUT modality dropout.
#    Shows that robustness to missing sequences comes from the training strategy, not the architecture.
RUN_ABLATION = False
if RUN_ABLATION:
    !python scripts/train.py --cache /kaggle/working/cache --out /kaggle/working/runs/no_moddrop --epochs 20 --no-modality-dropout
    !python scripts/evaluate.py --cache /kaggle/working/cache --ckpt /kaggle/working/runs/no_moddrop/best.pt --out /kaggle/working/runs/no_moddrop/results --skip-corruptions --no-hd95
    a = pd.read_csv("/kaggle/working/runs/base/results/subsets.csv").set_index("subset")["dice_WT"]
    b = pd.read_csv("/kaggle/working/runs/no_moddrop/results/subsets.csv").set_index("subset")["dice_WT"]
    display(pd.DataFrame({"with_modality_dropout": a, "without": b}).round(3))

In [ ]:
# 8) Package everything for download: checkpoints, results, one raw test case for the app demo
import json, shutil, os, glob
split = json.load(open("/kaggle/working/cache/split.json"))
demo_id = split["test"][0]
src = glob.glob(f"{RAW}/{demo_id}")[0]
os.makedirs("/kaggle/working/demo_case", exist_ok=True)
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), "/kaggle/working/demo_case/")
print("demo case:", demo_id, os.listdir("/kaggle/working/demo_case"))
!cd /kaggle/working && zip -q -r neuroscope_artifacts.zip runs demo_case && ls -lh neuroscope_artifacts.zip
print("Download neuroscope_artifacts.zip from the Output tab, unzip into your repo root, then: streamlit run app/app.py")